# STEP 16. POI 배정·지표와 분석 표

**목적**: 회귀에 바로 쓸 표를 만든다. POI(Advan) 파일이 있으면 결과변수까지 채우고, 없으면 배정 규칙만 합성 점으로 점검한다.

**POI 배정 규칙** (STEP 13의 상업 영역 정의와 같다)
1. 공원 부지 안의 POI는 뺀다.
2. POI를 가장 가까운 가로 표본점에 붙인다(`FRONT_DEPTH` = 50 m 이내).
3. 그 표본점이 공원 가장자리에서 길을 따라 100 m 안이면 상업 영역의 POI다.

| `rule` | 어느 공원·군집에 넣나 | 쓰임 |
|---|---|---|
| `nearest` | 그 지점까지 네트워크 거리가 가장 짧은 공원 하나. POI마다 한 번만 센다 | 주 분석 |
| `overlap` | 그 지점에 100 m 안으로 닿는 모든 공원. 한 군집 안에서는 한 번만 센다 | 민감도 |
| `buffer` | 직선 100 m 버퍼(가장 가까운 공원 기준으로 나눈 것) 안에 있으면 그 공원 | 비교 |

**POI 지표** (단위 = 공원 또는 군집)
- 업소 수, 가로 100 m당 업소 수(주 밀도), ha당 업소 수
- 업종 다양성: NAICS 분류의 Shannon 엔트로피, 업종 수
- 방문량: 합계, 업소당 평균, 가로 100 m당

**분석 표** (`outputs/{AREA}/analysis/`)
- `park_table.csv`: 규칙 × 공원. 공원 속성, 정의별 군집·묶임 여부, **D와 N의 일치 유형**, 가로·주변 여건 통제변수, (있으면) POI 지표
- `cluster_table_{def}.csv`: 규칙 × 군집. 군집 속성, 통제변수, (있으면) POI 지표
- `poi_table.parquet`: POI 단위(방문량의 POI 단위 모형용). POI 파일이 있을 때만 만든다.

In [1]:
# ===== 공통 설정 (모든 STEP 노트북에서 동일) =====
import os
import json
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)

# 분석 지역: "Brooklyn"(파일럿) 또는 "NYC"(전체). 환경변수 PARK_STUDY_AREA로도 지정 가능.
STUDY_AREA = os.environ.get("PARK_STUDY_AREA", "Brooklyn")

ROOT = Path.cwd()
DATA = ROOT / "Data"
PARKS_PATH = DATA / "NYCPark" / "Parks_Properties_20260923.geojson"
NET_DIR = DATA / "derived" / "network"        # STEP1 결과 (지역 무관, NYC+버퍼 전체)
AREA_DIR = DATA / "derived" / STUDY_AREA      # STEP2 이후 결과
OUT_DIR = ROOT / "outputs" / STUDY_AREA
for _d in (NET_DIR, AREA_DIR, OUT_DIR):
    _d.mkdir(parents=True, exist_ok=True)

CRS = 32618            # UTM 18N (m)
R_MAX = 3200           # 탐색 한계 (m) = 보로 네트워크 버퍼 폭
SPEED = 1.0            # speed_m_s=1 -> agg_seconds == 경로 길이(m)

BOROUGHS = {"M": "Manhattan", "X": "Bronx", "B": "Brooklyn", "Q": "Queens", "R": "Staten Island"}
AREA_BOROUGHS = ["B"] if STUDY_AREA == "Brooklyn" else list(BOROUGHS)

# 네트워크 변형: C = 가로 중심선 + 보행전용 경로 (주 분석), S = 보도 포함 전체 보행망 (민감도)
VARIANTS = ["C", "S"]
PRIMARY = "C"
SIDEWALK_FOOTWAYS = {"sidewalk", "crossing", "traffic_island"}

# 공원 필터 (사용자 결정): Flagship + 비공원성 필지 제외, 면적 상한 100 acres (계산은 200 상위집합)
EXCLUDE_TYPES = ["Flagship Park", "Operations", "Lot", "Undeveloped",
                 "Buildings/Institutions", "Cemetery", "Managed Sites"]
AREA_CAP = 100
AREA_CAP_SUPERSET = 200
AREA_CAP_GRID = [50, 100, 200]
RESTRICTED_TYPES = ["Garden", "Jointly Operated Playground"]            # 접근 제한 민감도
STREET_EMBEDDED_TYPES = ["Triangle/Plaza", "Strip", "Mall", "Parkway"]  # 가로 매몰형 민감도

# 공원-네트워크 attachment
ATTACH_BUFFER = {"C": 20.0, "S": 10.0}   # 경계 버퍼 (m): C는 중심선이 도로 폭 절반만큼 떨어져 있음
ATTACH_BUFFER_GRID = {"C": [15.0, 20.0, 30.0], "S": [5.0, 10.0, 20.0]}
INSIDE_SHARE = 0.9       # 공원 내부에 90% 이상 들어간 세그먼트는 attachment에서 제외 (통행은 허용)
K_SOURCES = 8            # 공원당 출발 attachment 최대 개수 (farthest-point sampling)
SMALL_COMPONENT = 200    # 이보다 작은 연결요소에만 붙은 공원은 'unresolved'

# 최종 묶음 공통 절차: 설정별 Louvain(γ) → 보로 단위 퍼콜레이션 필터 → 합의 ≥ 0.5 → 도보 10분 지름 분할
MAIN_GAMMA = 2
COMMERCIAL_BUFFER = 100   # 직선 100 m 버퍼 (비교용)

# 상업 영역 (주 정의): 공원 가장자리에서 보행망을 따라 100 m 이내의 가로
COMMERCIAL_DIST = 100     # 네트워크 거리 한계 (m)
FRONT_DEPTH = 50          # POI·필지를 가로에 붙이는 최대 거리 겸 영역 폴리곤 깊이 (m)
REACH_STEP = 2.5          # 가로 표본 간격 (m)
FRONT_KINDS = ["street"]  # 상점이 접하는 가로로 보는 세그먼트 종류

# 묶음 크기 제약: 묶음 안 모든 공원 쌍이 서로 도보 10분 이내 (보행망 최단거리, 1.33 m/s × 600 s ≈ 800 m)
WALK_LIMIT = 800
WALK_LIMIT_GRID = [400, 800, 1200]   # 5 / 10 / 15분 민감도

# N1 angular few-turn
N1_THETAS = [15, 45, 90, 180]
N1_RADII = [800, 1600, 3200]
N1_REL_Q = [0.1, 0.2]
DIST_BANDS = [0, 200, 400, 800, 1600, 3200]

# D walking distance (근접성 기준선): 보행망 최단거리 ≤ d 이면 연결
D_DISTS = [200, 400, 800]

# N2 street-axis continuity: 같은 연속 가로(stroke) 위, 두 공원 구간 사이 간격 ≤ lim
N2_ALONG = [400, 800, 1600]
# (큰길 속성 계산용) 각도 choice/NACH 반경과 상위 비율
N2_RADII = [800, 1600, 3200]
N2_TOP_P = [0.02, 0.05, 0.10]
STROKE_ANGLE = 30
STROKE_ANGLE_GRID = [20, 30, 45]
N2_MAX_ALONG = [3200, None]

# N3 excess few-turn field overlap
N3_THETAS = [45, 90]
N3_RADII = [800, 1600]
N3_TAU = [0.1, 0.2]
N3_KAPPA = [1.0, 1.25]

print(f"STUDY_AREA={STUDY_AREA}  boroughs={AREA_BOROUGHS}  AREA_DIR={AREA_DIR.relative_to(ROOT)}")

STUDY_AREA=NYC  boroughs=['M', 'X', 'B', 'Q', 'R']  AREA_DIR=Data/derived/NYC


In [2]:
import shapely
from scipy.spatial import cKDTree

CL_DIR, CT_DIR = OUT_DIR / "clusters", OUT_DIR / "controls"
AN_DIR = OUT_DIR / "analysis"
AN_DIR.mkdir(parents=True, exist_ok=True)
DEFS = ["D", "N1", "N2", "N3"]
RULES = ["nearest", "overlap", "buffer"]

# ---- POI 파일 설정 (Advan). 파일을 받으면 여기만 맞추면 된다 ----
POI_DIR = DATA / "Advan"                 # 이 폴더의 *.csv, *.csv.gz, *.parquet를 모두 읽는다
POI_COLS = {"id": "PLACEKEY", "lat": "LATITUDE", "lon": "LONGITUDE", "naics": "NAICS_CODE", "visits": "RAW_VISIT_COUNTS"}
NAICS_DIGITS = 4                          # 업종 다양성을 재는 NAICS 자릿수
# 상업 업소로 보는 NAICS 2자리 부문: 소매(44–45), 예술·오락(71), 숙박·음식(72), 기타 서비스(81)
COMMERCIAL_NAICS2 = ["44", "45", "71", "72", "81"]
EXCLUDE_NAICS3 = ["813", "814"]           # 종교·시민단체, 가사 서비스는 제외

## 16-1. 배정에 쓰는 자료 불러오기

In [3]:
memb = pd.read_csv(CL_DIR / "park_cluster_membership.csv").set_index("park_id")
PIDS = memb.index.tolist()
P = len(PIDS)
PIDX = pd.Series(np.arange(P), index=PIDS)
PARR = np.array(PIDS)
parks = gpd.read_parquet(NET_DIR / "parks.parquet").loc[PIDS]
cp = pd.read_parquet(AREA_DIR / "commercial_points.parquet")
cm = pd.read_parquet(AREA_DIR / "commercial_membership.parquet")
CP_TREE = cKDTree(cp[["x", "y"]].values)
CP_REACHED = cp.reached.values
CP_OWNER = PIDX.reindex(cp.owner.values).fillna(-1).astype(int).values
CM = pd.DataFrame({"pt": cm.point_id.values, "park": PIDX.loc[cm.park_id.values].values})
ALL_PARKS = gpd.read_file(PARKS_PATH).to_crs(CRS).geometry.make_valid().values
ALL_PARKS_TREE = shapely.STRtree(ALL_PARKS)


def load_zones(name):
    g = gpd.read_parquet(AREA_DIR / name).set_index("park_id").loc[PIDS]
    return g, np.array([x if x is not None else shapely.Polygon() for x in g.geometry.values], dtype=object)


PZ, ZONE_X = load_zones("park_zones.parquet")
_, ZONE_BUF = load_zones("park_zones_buffer.parquet")
ZONE_BUF_TREE = shapely.STRtree(ZONE_BUF)
SEG = pd.read_parquet(AREA_DIR / "segment_controls.parquet").drop(columns="geometry").set_index("seg_id")
STREET_COLS = ["ax_connectivity", "ax_int_n", "ax_int_r3", "ax_int_r5", "ax_int_r7", "ax_choice_log"] + \
              [c for c in SEG.columns if c.startswith(("nain_", "nach_"))]
CL = {d: np.unique(memb[f"{d}_cluster"].values, return_inverse=True) for d in DEFS}       # (군집 id 목록, 공원 → 군집 번호)

# 통제변수 표(STEP 14·15)와, 규칙별 밀도 분모(가로 길이 m, 영역 면적 m²)
street = pd.read_csv(CT_DIR / "park_street_controls.csv")
ctx = pd.read_csv(CT_DIR / "park_context_controls.csv")
cstreet = pd.read_csv(CT_DIR / "cluster_street_controls.csv")
cctx = pd.read_csv(CT_DIR / "cluster_context_controls.csv")
DEN = {r: (street[street.rule == r].set_index("park_id").street_len_m.reindex(PIDS).values,
           ctx[ctx.rule == r].set_index("park_id").area_ha.reindex(PIDS).fillna(0).values * 1e4) for r in RULES}
CDEN = {(r, d): (cstreet[(cstreet.rule == r) & (cstreet.definition == d)].set_index("cluster_id").street_len_m.reindex(CL[d][0]).values,
                 cctx[(cctx.rule == r) & (cctx.definition == d)].set_index("cluster_id").area_ha.reindex(CL[d][0]).fillna(0).values * 1e4)
        for r in RULES for d in DEFS}
print(f"{P:,} parks; street sample points {len(cp):,} ({CP_REACHED.sum():,} in a commercial area)")

2,057 parks; street sample points 1,532,439 (740,644 in a commercial area)


## 16-2. 배정 함수와 지표 함수

In [4]:
def assign_pois(x, y):
    """POI 좌표(프로젝트 좌표계) → POI별 배정 결과
    in_park, point(가장 가까운 가로 표본점, 없으면 -1), snap_m, in_area, park_nearest(공원 번호, 없으면 -1), n_parks, park_buffer"""
    xy = np.c_[x, y]
    pts = shapely.points(xy)
    in_park = np.zeros(len(xy), bool)
    in_park[np.unique(ALL_PARKS_TREE.query(pts, predicate="within")[0])] = True
    dist, idx = CP_TREE.query(xy, k=1, distance_upper_bound=FRONT_DEPTH)
    ok = np.isfinite(dist) & ~in_park
    point = np.where(ok, np.minimum(idx, len(cp) - 1), -1)
    in_area = ok & CP_REACHED[np.maximum(point, 0)]
    out = pd.DataFrame({"in_park": in_park, "point": point, "snap_m": np.where(ok, dist, np.nan), "in_area": in_area})
    out["park_nearest"] = np.where(in_area, CP_OWNER[np.maximum(point, 0)], -1)
    out["n_parks"] = np.where(in_area, cp.n_parks.values[np.maximum(point, 0)], 0)
    qi, zi = ZONE_BUF_TREE.query(pts, predicate="intersects")
    buf = pd.Series(zi, index=qi).groupby(level=0).first().reindex(range(len(xy))).fillna(-1).astype(int).values
    out["park_buffer"] = np.where(in_park, -1, buf)
    return out


def poi_park_pairs(a, rule):
    """배정 결과 → (POI 번호, 공원 번호) 쌍"""
    if rule == "nearest":
        k = np.where(a.park_nearest.values >= 0)[0]
        return k, a.park_nearest.values[k]
    if rule == "overlap":
        k = np.where(a.in_area.values)[0]
        m = pd.DataFrame({"poi": k, "pt": a.point.values[k]}).merge(CM, on="pt")
        return m.poi.values, m.park.values
    k = np.where(a.park_buffer.values >= 0)[0]
    return k, a.park_buffer.values[k]


def poi_metrics(poi, unit, n, naics, visits, street_len, area_m2):
    """(POI 번호, 단위 번호) 쌍 → 단위별 지표. 같은 POI는 한 단위 안에서 한 번만 센다."""
    pr = np.unique(np.c_[unit, poi], axis=0) if len(poi) else np.zeros((0, 2), int)
    u, k = pr[:, 0], pr[:, 1]
    out = pd.DataFrame({"poi_n": np.bincount(u, minlength=n).astype(float)})
    cat = pd.Series(naics[k]).str[:NAICS_DIGITS].values
    ct = pd.DataFrame({"u": u, "cat": cat}).groupby(["u", "cat"]).size().rename("c").reset_index()
    ct["p"] = ct.c / ct.groupby("u").c.transform("sum")
    ct["h"] = -ct.p * np.log(ct.p)
    out["poi_categories"] = ct.groupby("u").size().reindex(range(n)).fillna(0).values
    out["poi_diversity"] = ct.groupby("u").h.sum().reindex(range(n)).values            # Shannon 엔트로피 (업소가 없으면 결측)
    with np.errstate(all="ignore"):
        out["poi_per_100m"] = np.where(street_len > 0, out.poi_n / street_len * 100, np.nan)
        out["poi_per_ha"] = np.where(area_m2 > 0, out.poi_n / area_m2 * 1e4, np.nan)
        if visits is not None:
            v = np.bincount(u, weights=np.nan_to_num(visits[k]), minlength=n)
            out["visits"] = v
            out["visits_per_poi"] = np.where(out.poi_n > 0, v / out.poi_n.where(out.poi_n > 0), np.nan)
            out["visits_per_100m"] = np.where(street_len > 0, v / street_len * 100, np.nan)
    return out


def load_pois():
    """POI_DIR의 파일을 읽어 POI당 한 줄로 만든다(방문량은 합산). 파일이 없으면 None."""
    files = sorted(f for pat in ("*.csv", "*.csv.gz", "*.parquet") for f in POI_DIR.glob(pat)) if POI_DIR.exists() else []
    if not files:
        return None
    use = [c for c in POI_COLS.values()]
    frames = [pd.read_parquet(f, columns=use) if f.suffix == ".parquet" else pd.read_csv(f, usecols=use, dtype={POI_COLS["naics"]: str})
              for f in files]
    d = pd.concat(frames, ignore_index=True).rename(columns={v: k for k, v in POI_COLS.items()})
    d["naics"] = d.naics.astype(str).str.replace(r"\\.0$", "", regex=True)
    d = d.groupby("id").agg(lat=("lat", "first"), lon=("lon", "first"), naics=("naics", "first"), visits=("visits", "sum")).reset_index()
    keep = d.naics.str[:2].isin(COMMERCIAL_NAICS2) & ~d.naics.str[:3].isin(EXCLUDE_NAICS3)
    print(f"{len(files)} files, {len(d):,} POIs, {keep.sum():,} commercial (NAICS {', '.join(COMMERCIAL_NAICS2)})")
    d = d[keep].reset_index(drop=True)
    xy = gpd.GeoSeries(gpd.points_from_xy(d.lon, d.lat), crs=4326).to_crs(CRS)
    d["x"], d["y"] = xy.x.values, xy.y.values
    return d

## 16-3. 자체 점검 (합성 점)
진짜 POI가 아니라 **규칙이 뜻대로 동작하는지** 보는 점검이다. 합성 점의 결과는 어떤 표에도 저장하지 않는다.
- 가로 표본점 근처와 먼 곳에 임의 점을 찍는다.
- 확인: `nearest`는 POI마다 한 공원, `overlap` 합계 ≥ `nearest` 합계, `nearest`로 배정된 점은 그 공원의 영역 폴리곤 안, 군집 합계 = 공원 합계.

In [5]:
rng = np.random.default_rng(1)
n_syn = 60_000
base = cp[["x", "y"]].values[rng.integers(0, len(cp), n_syn)]
ang, rad = rng.uniform(0, 2 * np.pi, n_syn), rng.uniform(0, 70, n_syn)
syn_xy = base + np.c_[np.cos(ang) * rad, np.sin(ang) * rad]
syn_naics = rng.choice(["445110", "722511", "722513", "812112", "448140", "713940", "453998", "722410"], n_syn)
syn_visits = rng.lognormal(6, 1, n_syn)
A = assign_pois(syn_xy[:, 0], syn_xy[:, 1])
pairs = {r: poi_park_pairs(A, r) for r in RULES}
n_area = int(A.in_area.sum())
assert len(pairs["nearest"][0]) == n_area == len(np.unique(pairs["nearest"][0])), "nearest: 대상 POI마다 정확히 한 공원"
assert len(pairs["overlap"][0]) >= n_area and set(pairs["overlap"][0]) == set(pairs["nearest"][0])
k, pk = pairs["nearest"]
inside = shapely.intersects(ZONE_X[pk], shapely.points(syn_xy[k]))
print(f"synthetic points: {n_syn:,}; inside park land {A.in_park.sum():,}; snapped to a street {int((A.point >= 0).sum()):,}; "
      f"in a commercial area {n_area:,}; of those reached by 2+ parks {(A.n_parks >= 2).sum() / n_area:.1%}")
print(f"nearest-rule assignment falls inside that park's zone polygon: {inside.mean():.2%}")
assert inside.mean() >= 0.99
M = poi_metrics(*pairs["nearest"], P, syn_naics, syn_visits, *DEN["nearest"])
assert M.poi_n.sum() == n_area
ids, no = CL["N1"]
Mc = poi_metrics(pairs["nearest"][0], no[pairs["nearest"][1]], len(ids), syn_naics, syn_visits, *CDEN[("nearest", "N1")])
assert Mc.poi_n.sum() == M.poi_n.sum(), "군집 합계 = 공원 합계 (nearest)"
Mo = poi_metrics(*pairs["overlap"], P, syn_naics, syn_visits, *DEN["overlap"])
assert (Mo.poi_n >= M.poi_n).all(), "overlap은 공원마다 nearest 이상"
print("✓ assignment rules behave as specified (synthetic points only; nothing saved)")
pd.DataFrame({"nearest": M.poi_n.describe(), "overlap": Mo.poi_n.describe()}).round(1).loc[["mean", "50%", "max"]]

synthetic points: 60,000; inside park land 3,057; snapped to a street 56,162; in a commercial area 25,477; of those reached by 2+ parks 16.7%
nearest-rule assignment falls inside that park's zone polygon: 99.99%
✓ assignment rules behave as specified (synthetic points only; nothing saved)


,nearest,overlap
mean,12.4,15.0
50%,10.0,13.0
max,148.0,151.0


## 16-4. 실제 POI (파일이 있을 때)
`Data/Advan/`에 파일을 넣고 위쪽 `POI_COLS`를 파일의 열 이름에 맞추면, 아래에서 POI 단위 표와 공원·군집 지표가 만들어진다.

In [6]:
POI = load_pois()
PARK_POI, CLUSTER_POI = {}, {}
if POI is None:
    print(f"POI 파일 없음 ({POI_DIR.relative_to(ROOT)}/). 분석 표는 POI 지표 없이 만든다.")
else:
    A = assign_pois(POI.x.values, POI.y.values)
    naics, visits = POI.naics.values, POI.visits.values.astype(float)
    for r in RULES:
        k, pk = poi_park_pairs(A, r)
        PARK_POI[r] = poi_metrics(k, pk, P, naics, visits, *DEN[r])
        for d in DEFS:
            ids, no = CL[d]
            CLUSTER_POI[(r, d)] = poi_metrics(k, no[pk], len(ids), naics, visits, *CDEN[(r, d)])
    # POI 단위 표: 배정 결과 + 접한 가로의 값
    seg_of = cp.seg_id.values[np.maximum(A.point.values, 0)]
    pt = POI[["id", "naics", "visits"]].join(A)
    pt["park_nearest"] = np.where(A.park_nearest >= 0, PARR[np.maximum(A.park_nearest, 0)], None)
    pt["park_buffer"] = np.where(A.park_buffer >= 0, PARR[np.maximum(A.park_buffer, 0)], None)
    for d in DEFS:
        pt[f"{d}_cluster"] = memb[f"{d}_cluster"].reindex(pt.park_nearest).values
    pt = pt.join(SEG[STREET_COLS].reindex(seg_of).reset_index(drop=True).where(A.point.values[:, None] >= 0))
    pt[pt.in_area | (A.park_buffer.values >= 0)].to_parquet(AN_DIR / "poi_table.parquet", index=False)
    print(f"POIs in a commercial area: {int(A.in_area.sum()):,} of {len(POI):,}; saved poi_table.parquet")

POI 파일 없음 (Data/Advan/). 분석 표는 POI 지표 없이 만든다.


## 16-5. 공원 단위 분석 표
**D와 N의 일치 유형**: 같은 공원이 보행거리 정의(D)와 길 모양 정의(N1·N2·N3)에서 각각 묶였는지를 네 가지로 나눈다. "N에만 묶임"과 "D에만 묶임"의 비교가, 거리 말고 길 모양이 더하는 것이 있는지를 보는 핵심 대비다.

In [7]:
base = pd.DataFrame({"park_id": PIDS, "name": parks["name"].values, "parent_id": parks.parent_id.values,
                     "typecategory": parks.typecategory.values, "borough": parks.borough.values})
for d in DEFS:
    size = memb.groupby(f"{d}_cluster").size()
    base[f"{d}_cluster"] = memb[f"{d}_cluster"].values
    base[f"{d}_in_group"] = memb[f"{d}_in_group"].values
    base[f"{d}_cluster_size"] = size.reindex(memb[f"{d}_cluster"]).values
    base[f"{d}_status"] = memb[f"{d}_status"].values
for d in ["N1", "N2", "N3"]:
    base[f"D_vs_{d}"] = np.select([base.D_in_group & base[f"{d}_in_group"], base.D_in_group & ~base[f"{d}_in_group"],
                                   ~base.D_in_group & base[f"{d}_in_group"]], ["both", "D only", f"{d} only"], "neither")
frames = []
for r in RULES:
    t = base.copy()
    t.insert(0, "rule", r)
    t["zone_street_len_m"], t["zone_area_m2"] = DEN[r]
    t["no_street_reach"] = PZ.no_street_reach.values
    s = street[street.rule == r].set_index("park_id").drop(columns=["rule", "street_len_m"]).reindex(PIDS).reset_index(drop=True)
    c = ctx[ctx.rule == r].set_index("park_id").drop(columns="rule").reindex(PIDS).reset_index(drop=True)
    t = pd.concat([t, s.rename(columns={"basis": "street_basis"}), c], axis=1)
    if r in PARK_POI:
        t = pd.concat([t, PARK_POI[r]], axis=1)
    frames.append(t)
PARK_TABLE = pd.concat(frames, ignore_index=True)
PARK_TABLE.to_csv(AN_DIR / "park_table.csv", index=False)
assert len(PARK_TABLE) == len(RULES) * P and PARK_TABLE.columns.is_unique
print(f"park_table.csv: {PARK_TABLE.shape[0]:,} rows × {PARK_TABLE.shape[1]} columns")
pd.DataFrame({d: base[f"D_vs_{d}"].value_counts() for d in ["N1", "N2", "N3"]}).fillna(0).astype(int)

park_table.csv: 6,171 rows × 120 columns


,N1,N2,N3
D only,329,243,1187
N1 only,233,0,0
N2 only,0,181,0
N3 only,0,0,26
both,945,1031,87
neither,550,602,757


## 16-6. 군집 단위 분석 표 (정의별)

In [8]:
CLUSTER_TABLE = {}
for d in DEFS:
    ids, no = CL[d]
    attr = pd.read_csv(CL_DIR / f"clusters_{d}.csv").set_index("cluster_id").loc[ids].reset_index()
    attr = attr.rename(columns={"nach_1600_mean": "attach_nach_1600_mean"})      # 가로 통제변수의 nach_1600_mean(영역 평균)과 구분
    frames = []
    for r in RULES:
        t = attr.copy()
        t.insert(0, "rule", r)
        s = cstreet[(cstreet.rule == r) & (cstreet.definition == d)].set_index("cluster_id").drop(columns=["rule", "definition"]).reindex(ids)
        c = cctx[(cctx.rule == r) & (cctx.definition == d)].set_index("cluster_id").drop(columns=["rule", "definition"]).reindex(ids)
        t["zone_street_len_m"] = s.street_len_m.values
        t = pd.concat([t, s.drop(columns="street_len_m").rename(columns={"basis": "street_basis"}).reset_index(drop=True), c.reset_index(drop=True)], axis=1)
        if (r, d) in CLUSTER_POI:
            t = pd.concat([t, CLUSTER_POI[(r, d)]], axis=1)
        frames.append(t)
    CLUSTER_TABLE[d] = pd.concat(frames, ignore_index=True)
    assert CLUSTER_TABLE[d].columns.is_unique and len(CLUSTER_TABLE[d]) == len(RULES) * len(ids)
    CLUSTER_TABLE[d].to_csv(AN_DIR / f"cluster_table_{d}.csv", index=False)
    print(f"cluster_table_{d}.csv: {len(CLUSTER_TABLE[d]):,} rows × {CLUSTER_TABLE[d].shape[1]} columns")
pd.concat({d: CLUSTER_TABLE[d][CLUSTER_TABLE[d].rule == "nearest"].groupby("is_singleton")[
    ["zone_street_len_m", "pop_density", "job_density", "commercial_zoning_share", "ax_int_r3_mean", "nach_1600_mean"]].median().round(2)
    for d in DEFS}, names=["definition"])

cluster_table_D.csv: 3,396 rows × 121 columns


cluster_table_N1.csv: 3,657 rows × 121 columns


cluster_table_N2.csv: 3,522 rows × 121 columns


cluster_table_N3.csv: 5,973 rows × 121 columns


zone_street_len_m  pop_density  job_density  commercial_zoning_share  ax_int_r3_mean  nach_1600_mean
definition is_singleton                                                                                                      
D          False                   2429.82       209.25        35.36                     0.11            3.38            1.11
           True                     878.95       120.31        23.37                     0.05            3.30            1.11
N1         False                   2539.90       204.34        39.35                     0.13            3.56            1.12
           True                     748.77       116.75        21.04                     0.04            3.03            1.09
N2         False                   2776.47       210.83        38.58                     0.13            3.37            1.12
           True                     791.84       117.51        21.98                     0.04            3.24            1.10
N3         False                    678.30       249.27        32.84                     0.11            4.15            1.13
           True                     747.35       172.81        29.41                     0.08            3.35            1.11

## 16-7. 통제변수 후보의 다중공선성 (VIF)
공원 단위 표(`nearest` 규칙)에서 통제변수 후보의 VIF를 본다. 10을 넘으면 같이 넣지 않는다. 5를 넘으면 주의한다.

In [9]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

CAND = ["ax_int_r3_mean", "nach_1600_mean", "pop_density", "job_density", "median_income", "commercial_zoning_share",
        "floor_area_ratio", "com_floor_share", "n400_pop", "n400_jobs", "subway_dist_m", "parkland_400m_ha", "park_acres"]
X = PARK_TABLE[PARK_TABLE.rule == "nearest"][CAND].dropna()
Z = (X - X.mean()) / X.std()
Z.insert(0, "const", 1.0)
VIF = pd.Series([variance_inflation_factor(Z.values, k) for k in range(1, Z.shape[1])], index=CAND, name="VIF").round(2).sort_values(ascending=False)
VIF.to_csv(AN_DIR / "control_vif.csv")
print(f"{len(X):,} parks with all candidate controls present")
VIF.to_frame()

1,927 parks with all candidate controls present


,VIF
n400_jobs,3.95
job_density,3.84
pop_density,2.73
floor_area_ratio,2.62
n400_pop,2.62
park_acres,2.17
parkland_400m_ha,2.15
ax_int_r3_mean,1.68
commercial_zoning_share,1.52
nach_1600_mean,1.40


## 16-8. 변수 설명

In [10]:
has_poi = bool(PARK_POI)
readme = f"""# 분석 표 (STEP 16)

회귀에 바로 쓰는 표다. 통제변수의 뜻은 `../controls/README.md`(가로 구조)와 `../controls/README_context.md`(주변 여건)를, 군집 속성은 `../clusters/README.md`를 본다.

POI 지표: {"포함됨" if has_poi else "아직 없음. `Data/Advan/`에 파일을 넣고 STEP 16을 다시 실행하면 채워진다."}

## 파일
| 파일 | 단위 | 내용 |
|---|---|---|
| `park_table.csv` | 규칙 × 공원 ({P:,}곳) | 공원 속성, 정의별 군집, D와 N의 일치 유형, 통제변수{", POI 지표" if has_poi else ""} |
| `cluster_table_<def>.csv` | 규칙 × 군집 | 군집 속성(STEP 13), 통제변수{", POI 지표" if has_poi else ""} |
| `control_vif.csv` | | 통제변수 후보의 VIF (공원 단위, `nearest`) |
{"| `poi_table.parquet` | POI | 배정 결과와 접한 가로의 값 (POI 단위 모형용) |" if has_poi else ""}

## `rule`
| 값 | 상업 영역 | POI 배정 |
|---|---|---|
| `nearest` (주 분석) | 공원 가장자리에서 길을 따라 {COMMERCIAL_DIST} m 안의 가로 | 네트워크 거리가 가장 짧은 공원 하나 |
| `overlap` (민감도) | 같음 | 닿는 모든 공원. 한 군집 안에서는 한 번만 셈 |
| `buffer` (비교) | 직선 {COMMERCIAL_BUFFER} m 버퍼 − 공원 부지 | 직선 거리로 가장 가까운 공원 |

## 공원 표의 열
| 열 | 뜻 |
|---|---|
| `park_id`, `name`, `parent_id`, `typecategory`, `borough` | 공원 속성. 긴 선형 공원은 구간으로 나뉘어 있고 `parent_id`가 원래 공원이다 |
| `<def>_cluster`, `<def>_in_group`, `<def>_cluster_size`, `<def>_status` | 정의(D·N1·N2·N3)별 군집 id, 묶음에 속하는지, 군집의 공원 수, 상태 |
| `D_vs_N1`, `D_vs_N2`, `D_vs_N3` | D와 N의 일치 유형: `both`, `D only`, `N? only`, `neither` |
| `zone_street_len_m`, `zone_area_m2` | 그 규칙에서 공원에 배정된 가로 길이와 영역 면적. 밀도의 분모 |
| `no_street_reach` | 길을 따라 {COMMERCIAL_DIST} m 안에 가로가 없는 공원 |
| `ax_*`, `nain_*`, `nach_*`, `top10_*`, `street_basis` | 가로 구조 통제변수 (STEP 14) |
| `area_ha` … `parkland_800m_ha` | 주변 여건 통제변수 (STEP 15) |
| `poi_n`, `poi_per_100m`, `poi_per_ha` | 업소 수, 가로 100 m당, ha당 (POI 파일이 있을 때) |
| `poi_categories`, `poi_diversity` | NAICS {NAICS_DIGITS}자리 업종 수, Shannon 엔트로피 |
| `visits`, `visits_per_poi`, `visits_per_100m` | 방문량 합계, 업소당, 가로 100 m당 |

## 쓸 때 주의
- `nearest` 규칙에서는 가로가 전부 더 가까운 다른 공원에 배정돼 영역이 비는 공원이 있다(`zone_street_len_m` = 0). 공원 단위 모형에서는 이 공원들을 빼거나 `overlap` 규칙과 비교한다.
- 군집 표의 `street_len_m`은 STEP 13 값(`nearest`)이고, `zone_street_len_m`이 그 행의 규칙에 맞는 길이다.
- 군집 표의 `attach_nach_1600_mean`은 구성 공원이 붙은 가로의 NACH 평균(STEP 13)이고, `nach_1600_mean`은 상업 영역 가로의 평균(STEP 14)이다.
- 상업 업소의 범위는 노트북 위쪽 `COMMERCIAL_NAICS2`, `EXCLUDE_NAICS3`에서 정한다(지금은 소매·예술오락·숙박음식·기타 서비스).
"""
(AN_DIR / "README.md").write_text(readme)
print(sorted(f.name for f in AN_DIR.iterdir()))

['README.md', 'cluster_table_D.csv', 'cluster_table_N1.csv', 'cluster_table_N2.csv', 'cluster_table_N3.csv', 'control_vif.csv', 'park_table.csv']
